# Taller práctico de análisis de datos con Pandas
## Consumo de Energía y Servicios Públicos Inteligentes

**Empresa:** Energía de Occidente  


## 1. Importación y verificación de estructura

Se importa Pandas y se carga el CSV en el DataFrame `df_energia`. `index_col=False` evita interpretar una columna como índice y `low_memory=False` ayuda a que Pandas infiera los tipos de datos de forma consistente.

In [70]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Funciona al ejecutar el notebook desde la raíz del proyecto.
RUTA_CSV = Path("datos") / "consumo_energia_iot.csv"
if not RUTA_CSV.exists():
    # Alternativa útil si el notebook se abre desde una carpeta distinta.
    RUTA_CSV = Path.cwd() / "datos" / "consumo_energia_iot.csv"

df_energia = pd.read_csv(RUTA_CSV, index_col=False, low_memory=False)

print("Dimensiones:", df_energia.shape)
print("\nColumnas:")
print(df_energia.columns.tolist())
print("\nTipos de datos:")
display(df_energia.dtypes.to_frame("Tipo de dato"))
print("\nPrimeras cinco filas:")
display(df_energia.head())

Dimensiones: (1012, 8)

Columnas:
['ID_Medidor', 'Municipio', 'Estrato', 'Consumo_kWh', 'Voltaje_Promedio', 'Horas_Pico_kWh', 'Tipo_Medidor', 'Estado_Pago']

Tipos de datos:


,Tipo de dato
ID_Medidor,str
Municipio,str
Estrato,str
Consumo_kWh,float64
Voltaje_Promedio,float64
Horas_Pico_kWh,float64
Tipo_Medidor,str
Estado_Pago,str



Primeras cinco filas:


,ID_Medidor,Municipio,Estrato,Consumo_kWh,Voltaje_Promedio,Horas_Pico_kWh,Tipo_Medidor,Estado_Pago
0,MED-5001,Villamaría,Estrato 3,130.5,118.0,32.74,Monofásico Digital,Al Día
1,MED-5002,Palestina,Estrato 2,175.0,120.5,19.97,Bifásico Inteligente,Al Día
2,MED-5003,Palestina,Estrato 1,220.0,118.0,11.71,Bifásico Inteligente,Mora
3,MED-5004,Neira,Estrato 4,130.5,128.0,64.26,Monofásico Digital,Al Día
4,MED-5005,Neira,Estrato 3,175.0,120.5,14.13,Monofásico Digital,Mora


## 2. Selección unidimensional y proyección tabular

- `consumo_serie` es una `Series` con la columna `Consumo_kWh`.
- `vista_ubicacion` contiene únicamente `Municipio`, `Estrato` y `Voltaje_Promedio`.

In [71]:
consumo_serie = df_energia["Consumo_kWh"]
vista_ubicacion = df_energia[["Municipio", "Estrato", "Voltaje_Promedio"]]

print("Tipo de consumo_serie:", type(consumo_serie))
display(consumo_serie.head())
display(vista_ubicacion.head())

Tipo de consumo_serie: <class 'pandas.Series'>


0    130.5
1    175.0
2    220.0
3    130.5
4    175.0
Name: Consumo_kWh, dtype: float64

,Municipio,Estrato,Voltaje_Promedio
0,Villamaría,Estrato 3,118.0
1,Palestina,Estrato 2,120.5
2,Palestina,Estrato 1,118.0
3,Neira,Estrato 4,128.0
4,Neira,Estrato 3,120.5


## 3. Consulta de lectura específica con `.loc[]`

Se extrae la fila cuyo **índice de Pandas es 150**. El índice comienza en cero.

In [72]:
fila_indice_150 = df_energia.loc[150]
display(fila_indice_150.to_frame(name="Valor"))

,Valor
ID_Medidor,MED-5151
Municipio,Neira
Estrato,Estrato 2
Consumo_kWh,130.5
Voltaje_Promedio,112.5
Horas_Pico_kWh,31.42
Tipo_Medidor,Trifásico Industrial
Estado_Pago,Financiado


## 4. Depuración de registros duplicados

`drop_duplicates()` elimina filas que sean idénticas en todas sus columnas. Se conserva el primer ejemplar de cada fila repetida.

In [73]:
filas_antes = len(df_energia)
df_sin_duplicados = df_energia.drop_duplicates().copy()
filas_despues = len(df_sin_duplicados)

print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Filas duplicadas eliminadas: {filas_antes - filas_despues}")
print(f"Identificadores únicos antes: {df_energia['ID_Medidor'].nunique()}")

Filas antes: 1012
Filas después: 1000
Filas duplicadas eliminadas: 12
Identificadores únicos antes: 1000


## 5. Filtrado booleano simple: consumo superior a 250 kWh

Se seleccionan los registros con `Consumo_kWh > 250.0`.

In [74]:
consumo_elevado = df_sin_duplicados[df_sin_duplicados["Consumo_kWh"] > 250.0]
print("Cantidad de registros:", len(consumo_elevado))
display(consumo_elevado.head())

Cantidad de registros: 189


,ID_Medidor,Municipio,Estrato,Consumo_kWh,Voltaje_Promedio,Horas_Pico_kWh,Tipo_Medidor,Estado_Pago
13,MED-5014,Neira,Estrato 4,290.5,112.5,69.74,Bifásico Inteligente,Al Día
16,MED-5017,Palestina,Estrato 1,290.5,118.0,12.81,Trifásico Industrial,Al Día
29,MED-5030,Manizales,Estrato 1,360.0,122.0,74.21,Monofásico Digital,Financiado
30,MED-5031,Neira,Estrato 2,290.5,125.5,6.57,Bifásico Inteligente,Financiado
40,MED-5041,Manizales,Estrato 2,290.5,122.0,66.32,Trifásico Industrial,Al Día


## 6. Filtrado categórico con `.isin()`

Se recuperan los medidores de los municipios de Chinchiná y Villamaría.

In [75]:
municipios_objetivo = ["Chinchiná", "Villamaría"]
medidores_municipios = df_sin_duplicados[
    df_sin_duplicados["Municipio"].isin(municipios_objetivo)
]
print("Cantidad de registros:", len(medidores_municipios))
display(medidores_municipios.head())

Cantidad de registros: 390


,ID_Medidor,Municipio,Estrato,Consumo_kWh,Voltaje_Promedio,Horas_Pico_kWh,Tipo_Medidor,Estado_Pago
0,MED-5001,Villamaría,Estrato 3,130.5,118.0,32.74,Monofásico Digital,Al Día
5,MED-5006,Chinchiná,Estrato 4,175.0,120.5,50.49,Bifásico Inteligente,Financiado
8,MED-5009,Villamaría,Estrato 2,130.5,122.0,45.63,Bifásico Inteligente,Mora
14,MED-5015,Chinchiná,Estrato 3,130.5,118.0,5.48,Bifásico Inteligente,Al Día
15,MED-5016,Chinchiná,Estrato 3,95.0,118.0,49.84,Monofásico Digital,Al Día


## 7. Filtrado por intervalo con `.between()`

El intervalo de voltaje óptimo es inclusivo: de 118.0 V a 122.0 V. Los valores nulos no cumplen la condición.

In [76]:
voltaje_optimo = df_sin_duplicados[
    df_sin_duplicados["Voltaje_Promedio"].between(118.0, 122.0, inclusive="both")
]
print("Cantidad de registros:", len(voltaje_optimo))
display(voltaje_optimo.head())

Cantidad de registros: 705


,ID_Medidor,Municipio,Estrato,Consumo_kWh,Voltaje_Promedio,Horas_Pico_kWh,Tipo_Medidor,Estado_Pago
0,MED-5001,Villamaría,Estrato 3,130.5,118.0,32.74,Monofásico Digital,Al Día
1,MED-5002,Palestina,Estrato 2,175.0,120.5,19.97,Bifásico Inteligente,Al Día
2,MED-5003,Palestina,Estrato 1,220.0,118.0,11.71,Bifásico Inteligente,Mora
4,MED-5005,Neira,Estrato 3,175.0,120.5,14.13,Monofásico Digital,Mora
5,MED-5006,Chinchiná,Estrato 4,175.0,120.5,50.49,Bifásico Inteligente,Financiado


## 8. Consulta con `.query()`

Se filtran inmuebles clasificados como `Comercial` cuyo estado de pago sea `Mora`.

In [77]:
comercial_en_mora = df_sin_duplicados.query(
    "Estrato == 'Comercial' and Estado_Pago == 'Mora'"
)
print("Cantidad de registros:", len(comercial_en_mora))
display(comercial_en_mora.head())

Cantidad de registros: 32


,ID_Medidor,Municipio,Estrato,Consumo_kWh,Voltaje_Promedio,Horas_Pico_kWh,Tipo_Medidor,Estado_Pago
58,MED-5059,Villamaría,Comercial,290.5,122.0,75.92,Trifásico Industrial,Mora
84,MED-5085,Manizales,Comercial,220.0,118.0,38.27,Monofásico Digital,Mora
95,MED-5096,Chinchiná,Comercial,130.5,128.0,47.36,Monofásico Digital,Mora
115,MED-5116,Villamaría,Comercial,175.0,120.5,74.32,Monofásico Digital,Mora
129,MED-5130,Chinchiná,Comercial,95.0,120.5,44.48,Monofásico Digital,Mora


## 9. Filtrado compuesto con métodos de texto y operador `&`

Se buscan lecturas cuyo `Tipo_Medidor` comience por la letra **B** y cuyas `Horas_Pico_kWh` sean mayores que 40.0. Se usa `.str.startswith()` para la condición de texto.

In [78]:
condicion_tipo_b = df_sin_duplicados["Tipo_Medidor"].str.startswith("B", na=False)
condicion_horas_pico = df_sin_duplicados["Horas_Pico_kWh"] > 40.0
medidores_b_pico_alto = df_sin_duplicados[condicion_tipo_b & condicion_horas_pico]

print("Cantidad de registros:", len(medidores_b_pico_alto))
display(medidores_b_pico_alto.head())

Cantidad de registros: 180


,ID_Medidor,Municipio,Estrato,Consumo_kWh,Voltaje_Promedio,Horas_Pico_kWh,Tipo_Medidor,Estado_Pago
5,MED-5006,Chinchiná,Estrato 4,175.0,120.5,50.49,Bifásico Inteligente,Financiado
7,MED-5008,Palestina,Estrato 1,220.0,120.5,51.93,Bifásico Inteligente,Mora
8,MED-5009,Villamaría,Estrato 2,130.5,122.0,45.63,Bifásico Inteligente,Mora
13,MED-5014,Neira,Estrato 4,290.5,112.5,69.74,Bifásico Inteligente,Al Día
17,MED-5018,Palestina,Estrato 2,NaN,120.5,54.42,Bifásico Inteligente,Mora


## 10. Tratamiento e imputación de valores nulos

- Los nulos de `Consumo_kWh` se reemplazan por la media de la columna calculada después de retirar duplicados.
- Los nulos de `Voltaje_Promedio` se reemplazan por el valor nominal de `120.0` voltios.

Se crea `df_limpio` para conservar `df_energia` como copia de los datos importados.

In [79]:
df_limpio = df_sin_duplicados.copy()

media_consumo = df_limpio["Consumo_kWh"].mean()
df_limpio["Consumo_kWh"] = df_limpio["Consumo_kWh"].fillna(media_consumo)
df_limpio["Voltaje_Promedio"] = df_limpio["Voltaje_Promedio"].fillna(120.0)

print(f"Media usada para imputar Consumo_kWh: {media_consumo:.2f} kWh")
print("\nNulos después de imputar:")
display(df_limpio[["Consumo_kWh", "Voltaje_Promedio"]].isna().sum().to_frame("Nulos"))

Media usada para imputar Consumo_kWh: 193.13 kWh

Nulos después de imputar:


,Nulos
Consumo_kWh,0
Voltaje_Promedio,0


## 11. Transformación condicional con `.apply()`

La función asigna:
- **Bajo:** consumo menor que 120 kWh.
- **Medio:** consumo entre 120 y 250 kWh, incluidos ambos límites.
- **Alto:** consumo mayor que 250 kWh.

Como los nulos de consumo ya se imputaron, cada fila recibe un nivel.

In [80]:
def clasificar_consumo(consumo):
    if consumo < 120:
        return "Bajo"
    elif consumo <= 250:
        return "Medio"
    else:
        return "Alto"

df_limpio["Nivel_Consumo"] = df_limpio["Consumo_kWh"].apply(clasificar_consumo)

display(df_limpio[["ID_Medidor", "Consumo_kWh", "Nivel_Consumo"]].head())
print("\nCantidad por nivel:")
display(df_limpio["Nivel_Consumo"].value_counts().to_frame("Cantidad"))

,ID_Medidor,Consumo_kWh,Nivel_Consumo
0,MED-5001,130.5,Medio
1,MED-5002,175.0,Medio
2,MED-5003,220.0,Medio
3,MED-5004,130.5,Medio
4,MED-5005,175.0,Medio



Cantidad por nivel:


,Cantidad
Nivel_Consumo,
Medio,651
Alto,189
Bajo,160


## 12. Renombrado y reordenamiento de columnas

Se renombran `Consumo_kWh` como `Energia_kWh` y `Tipo_Medidor` como `Tecnologia`. Luego se colocan `ID_Medidor` y `Nivel_Consumo` al principio de la tabla.

In [81]:
df_energia_final = df_limpio.rename(
    columns={
        "Consumo_kWh": "Energia_kWh",
        "Tipo_Medidor": "Tecnologia"
    }
)

columnas_inicio = ["ID_Medidor", "Nivel_Consumo"]
otras_columnas = [col for col in df_energia_final.columns if col not in columnas_inicio]
df_energia_final = df_energia_final[columnas_inicio + otras_columnas]

display(df_energia_final.head())
print("\nOrden final de columnas:")
print(df_energia_final.columns.tolist())

,ID_Medidor,Nivel_Consumo,Municipio,Estrato,Energia_kWh,Voltaje_Promedio,Horas_Pico_kWh,Tecnologia,Estado_Pago
0,MED-5001,Medio,Villamaría,Estrato 3,130.5,118.0,32.74,Monofásico Digital,Al Día
1,MED-5002,Medio,Palestina,Estrato 2,175.0,120.5,19.97,Bifásico Inteligente,Al Día
2,MED-5003,Medio,Palestina,Estrato 1,220.0,118.0,11.71,Bifásico Inteligente,Mora
3,MED-5004,Medio,Neira,Estrato 4,130.5,128.0,64.26,Monofásico Digital,Al Día
4,MED-5005,Medio,Neira,Estrato 3,175.0,120.5,14.13,Monofásico Digital,Mora



Orden final de columnas:
['ID_Medidor', 'Nivel_Consumo', 'Municipio', 'Estrato', 'Energia_kWh', 'Voltaje_Promedio', 'Horas_Pico_kWh', 'Tecnologia', 'Estado_Pago']


## 13. Agregación por municipio con `.groupby()` y `.agg()`

Para cada municipio se calcula el consumo promedio (`Energia_kWh`) y el total acumulado de `Horas_Pico_kWh`.

In [82]:
resumen_municipio = (
    df_energia_final
    .groupby("Municipio")
    .agg(
        Consumo_promedio_kWh=("Energia_kWh", "mean"),
        Horas_pico_total_kWh=("Horas_Pico_kWh", "sum")
    )
    .sort_values("Consumo_promedio_kWh", ascending=False)
)

display(resumen_municipio.round(2))

,Consumo_promedio_kWh,Horas_pico_total_kWh
Municipio,,
Manizales,199.55,8862.39
Chinchiná,193.03,8359.66
Neira,192.58,8975.47
Palestina,192.45,9101.80
Villamaría,187.75,8583.58


## 14. Métricas de rango personalizadas agrupadas por estrato

La función `rango(series)` calcula el máximo menos el mínimo. Se aplica con `.agg()` para obtener el rango del consumo y del voltaje promedio en cada estrato.

In [83]:
def rango(series):
    return series.max() - series.min()

resumen_rango_estrato = (
    df_energia_final
    .groupby("Estrato")
    .agg(
        Rango_consumo_kWh=("Energia_kWh", rango),
        Rango_voltaje_V=("Voltaje_Promedio", rango)
    )
)

display(resumen_rango_estrato.round(2))

,Rango_consumo_kWh,Rango_voltaje_V
Estrato,,
Comercial,385.0,15.5
Estrato 1,385.0,15.5
Estrato 2,385.0,15.5
Estrato 3,385.0,15.5
Estrato 4,385.0,15.5


## 15. Agrupamiento multinivel por municipio y estrato

Se calcula el promedio de energía y el promedio de voltaje para cada combinación de `Municipio` y `Estrato`.

In [84]:
resumen_municipio_estrato = (
    df_energia_final
    .groupby(["Municipio", "Estrato"])
    .agg(
        Energia_promedio_kWh=("Energia_kWh", "mean"),
        Voltaje_promedio_V=("Voltaje_Promedio", "mean")
    )
    .sort_index()
)

display(resumen_municipio_estrato.round(2))

Energia_promedio_kWh  Voltaje_promedio_V
Municipio  Estrato                                            
Chinchiná  Comercial                188.30              121.02
           Estrato 1                184.51              120.39
           Estrato 2                205.06              120.78
           Estrato 3                196.75              119.98
           Estrato 4                186.19              120.58
Manizales  Comercial                182.44              120.78
           Estrato 1                220.48              118.92
           Estrato 2                186.84              120.60
           Estrato 3                217.35              120.75
           Estrato 4                185.32              121.40
Neira      Comercial                182.25              120.37
           Estrato 1                187.12              119.71
           Estrato 2                184.33              120.11
           Estrato 3                197.26              120.24
           Estrato 4                217.32              119.96
Palestina  Comercial                204.91              119.95
           Estrato 1                199.49              120.96
           Estrato 2                190.61              120.23
           Estrato 3                191.86              120.67
           Estrato 4                169.01              121.56
Villamaría Comercial                202.69              120.86
           Estrato 1                186.77              120.11
           Estrato 2                184.29              120.67
           Estrato 3                181.56              120.02
           Estrato 4                191.24              119.60

## 16. Comprobaciones finales

Se comprueba que el DataFrame final no tenga nulos en las dos columnas imputadas, que las filas duplicadas se hayan eliminado y que los nombres y el orden de las columnas sean los esperados.

In [85]:
print("Filas del DataFrame final:", len(df_energia_final))
print("Filas duplicadas en el resultado:", df_energia_final.duplicated().sum())
print("\nNulos en columnas imputadas:")
display(df_energia_final[["Energia_kWh", "Voltaje_Promedio"]].isna().sum().to_frame("Nulos"))
print("\nPrimeras columnas:", df_energia_final.columns[:2].tolist())

Filas del DataFrame final: 1000
Filas duplicadas en el resultado: 0

Nulos en columnas imputadas:


,Nulos
Energia_kWh,0
Voltaje_Promedio,0



Primeras columnas: ['ID_Medidor', 'Nivel_Consumo']
